In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px
import plotly.graph_objects as go
f = pd.read_csv("Mall_Customers.csv" )
f = f.dropna()
f["GenderNo"] = np.where(f["Gender"]== "Male" , 1 ,3 )
df = f[['GenderNo', "Annual_Income", "Spending_Score"]]
scaler = StandardScaler()
dfnew = scaler.fit_transform(df)
# decide k and run kmeans
scores = {}
for k in range(2,11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(dfnew)
    scores[k] = silhouette_score(dfnew, km.labels_)
best_k = max(scores, key=scores.get)
km = KMeans(n_clusters=best_k, n_init=10, random_state=42)
labels = km.fit_predict(dfnew)
spend_mean = df["Spending_Score"].mean()
income_mean = df["Annual_Income"].mean()
def label(row):
    hi_spend = row["Spending_Score"] > spend_mean
    low_spend = row["Spending_Score"] < spend_mean
    hi_salary = row["Annual_Income"] > income_mean
    low_salary = row["Annual_Income"] < income_mean
    Male  = row["GenderNo"] < 2
    Female  = row["GenderNo"] > 2
    if hi_spend and low_salary and Male:
        return "Male With High Spend and Low Income"
    if hi_spend and hi_salary and Male:
        return "Male With High Spend and High Income"
    if hi_spend and low_salary and Female:
        return "Female With High Spend and Low Income"
    if hi_spend and hi_salary and Female:
        return "Female With High Spend and High Income"
    if low_spend and low_salary and Male:
        return "Male With Low Spend and Low Income"
    if low_spend and hi_salary and Male:
        return "Male With Low Spend and High Income"
    if low_spend and low_salary and Female:
        return "Female With Low Spend and Low Income"
    if low_spend and hi_salary and Female:
        return "Female With Low Spend and High Income"
df['Label_Kmeans'] = labels
# eps could be 0.4 or 0.5
db = DBSCAN(eps=0.4, min_samples=5)
label1 = db.fit_predict(dfnew)
df["Gender"] = f["Gender"]
df["CustomerID"] = f["CustomerID"]
df["Label_DBSCAN"] = label1
df['About'] = df.apply(label, axis=1)
df.to_csv("new.csv")
def recco(text):
    rcc= []
    if "male" in text.lower():
        x = df[df["Gender"] == "Male"]["CustomerID"]
        x = x.tolist()
        rcc.extend(x)
    elif "female" in text.lower():
        x = df[df["Gender"] == "Female"]["CustomerID"]
        x = x.tolist()
        rcc.extend(x)
    elif "high" in text.lower():
        if "income" in text.lower():
            x = df[df["Annual_Income"] > income_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
        if "spend" in text.lower():
            x = df[df["Spending_Score"] > spend_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
    elif "low" in text.lower():
        if "income" in text.lower():
            x = df[df["Annual_Income"] < income_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
        if "spend" in text.lower():
            x = df[df["Spending_Score"] < spend_mean]["CustomerID"]
            x = x.tolist()
            rcc.extend(x)
    return rcc
recco("male")


[1, 2, 9, 11, 15, 16, 18, 19, 21, 22, 24, 26, 28, 31, 33, 34, 42, 43, 52, 54, 56, 58, 60, 61, 62, 65, 66, 69, 71, 75, 76, 78, 81, 82, 83, 86, 92, 93, 96, 99, 100, 103, 104, 105, 108, 109, 110, 111, 114, 121, 124, 127, 128, 129, 130, 131, 132, 135, 138, 139, 142, 145, 146, 147, 150, 151, 152, 157, 159, 163, 165, 167, 170, 171, 172, 173, 174, 177, 178, 179, 180, 183, 186, 188, 193, 198, 199, 200]


C:\Users\faiza\AppData\Local\Temp\ipykernel_43152\3897412503.py:51: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label_Kmeans'] = labels
C:\Users\faiza\AppData\Local\Temp\ipykernel_43152\3897412503.py:55: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Gender"] = f["Gender"]
